# [Chapter 7 — SIR_I Simulations, §7.3] Verification: Numerical vs Analytical Predictions

**Book:** *Essential Considerations for Modeling Epidemics* (Hyman/Qu/Xue, 2026)
**Chapter:** Chapter 7 — SIR_I Simulations
**Considerations developed:** 7 (equilibria + stability), 9 (practical fitting)
**Estimated runtime:** ≤ 1 minute

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/machyman/hyman2026essential/blob/main/python/notebooks/)

## What this notebook does
Cross-checks numerical trajectories against analytical predictions in three regimes: (1) early-outbreak exponential growth rate, (2) peak-time and peak-height scaling laws, (3) final-size relation. Demonstrates the book's verification protocol of running independent analytical and numerical estimates and asserting agreement to documented tolerance.

## What you should already know
Chapter 7 numerical-integration notebook.


## Setup


In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), '..', '..')))

import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import brentq
from shared import book_style, BOOK_COLORS, integrate_sir_i
from shared.parameters import baseline_chapter_07
from shared.seeds import set_seed_chapter_07
from shared.verification import assert_within_tolerance

set_seed_chapter_07()
book_style()


## Scenario 1: Early-outbreak growth rate

In the linearized regime, $I(t) \propto e^{rt}$ with $r = (R_0 - 1)/\tau_R$. We measure $r$ from the simulation and compare.

In [ ]:
params = baseline_chapter_07()
result = integrate_sir_i(params)
t, S, I = result['t'], result['S'], result['I']

R0_predicted = params['c_I'] * params['beta'] * params['tau_R']
r_predicted = (R0_predicted - 1) / params['tau_R']

# Fit slope of log I in early regime (5 < t < 20 days)
mask = (t > 5) & (t < 20)
slope, _ = np.polyfit(t[mask], np.log(I[mask]), 1)

print(f"Predicted growth rate r = (R_0 - 1)/tau_R = {r_predicted:.5f} per day")
print(f"Measured growth rate    = {slope:.5f} per day")
assert_within_tolerance(slope, r_predicted, rel_tol=0.02)
print("✅ Growth rate matches within 2%")


## Scenario 2: Peak-time scaling

For SIR_I without vital dynamics, the peak occurs when $S/N = 1/R_0$. The peak time depends on $R_0$ and initial conditions but obeys the relationship $S(t_{peak}) = 1/R_0$ exactly.

In [ ]:
peak_idx = I.argmax()
S_at_peak_numerical = S[peak_idx]
S_at_peak_predicted = 1.0 / R0_predicted

print(f"Predicted S/N at peak: 1/R_0 = {S_at_peak_predicted:.6f}")
print(f"Measured S/N at peak (from sim): {S_at_peak_numerical:.6f}")
assert_within_tolerance(S_at_peak_numerical, S_at_peak_predicted, rel_tol=0.005)
print("✅ Peak-time S(t_peak) = 1/R_0 verified within 0.5%")


## Scenario 3: Final-size relation

The transcendental final-size equation:
$$\ln \frac{S_0}{S_\infty} = R_0 (1 - S_\infty)$$

determines $S_\infty$ uniquely from $R_0$ and $S_0$. We solve it numerically and compare with the long-time asymptote of $S(t)$.

In [ ]:
S_0 = params['S0']

def final_size_equation(S_inf):
    return np.log(S_0 / S_inf) - R0_predicted * (1 - S_inf)

S_inf_analytical = brentq(final_size_equation, 1e-6, S_0 - 1e-6)
S_inf_numerical = S[-1]

print(f"Final-size analytical (transcendental):  {S_inf_analytical:.6f}")
print(f"Final-size numerical (S at t = {t[-1]:.0f}): {S_inf_numerical:.6f}")
attack_rate = 1 - S_inf_analytical
print(f"Final attack rate: {attack_rate*100:.2f}% of population ever infected")


In [ ]:
# Visualize all three verification scenarios
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

# Panel 1: log I early growth
ax = axes[0]
mask = (t > 0) & (t < 30)
ax.semilogy(t[mask], I[mask], color=BOOK_COLORS['infectious'], lw=1.5, label='I(t) numerical')
mask2 = (t > 5) & (t < 20)
fit_line = np.exp(slope * t[mask2] + np.log(I[mask2][0]) - slope * t[mask2][0])
ax.semilogy(t[mask2], fit_line, ls='--', color=BOOK_COLORS['neutral'], label=f'Fit: $r = {slope:.3f}$/day')
ax.set_xlabel('Time (days)')
ax.set_ylabel('I(t)')
ax.set_title(f'Scenario 1: growth rate\n(predicted: {r_predicted:.3f})')
ax.legend(fontsize=9)

# Panel 2: peak with S(t_peak) marker
ax = axes[1]
ax.plot(t, I, color=BOOK_COLORS['infectious'], lw=1.5, label='I(t)')
ax.axvline(t[peak_idx], ls=':', color=BOOK_COLORS['neutral'])
ax.plot([t[peak_idx]], [I[peak_idx]], 'o', color=BOOK_COLORS['infectious'], ms=8)
ax.set_xlabel('Time (days)')
ax.set_ylabel('I(t)')
ax.set_title(f'Scenario 2: peak\n(S = {S_at_peak_numerical:.3f}, predicted: 1/R_0 = {S_at_peak_predicted:.3f})')
ax.set_xlim(0, 200)

# Panel 3: final-size relation
ax = axes[2]
ax.plot(t, S, color=BOOK_COLORS['susceptible'], lw=1.5, label='S(t) numerical')
ax.axhline(S_inf_analytical, ls='--', color=BOOK_COLORS['neutral'], label=f'$S_\\infty$ analytical = {S_inf_analytical:.4f}')
ax.set_xlabel('Time (days)')
ax.set_ylabel('S(t)')
ax.set_title(f'Scenario 3: final size\n(attack rate: {attack_rate*100:.1f}%)')
ax.legend(fontsize=9)
ax.set_xlim(0, 200)

plt.tight_layout()
plt.show()


## What's next

Three independent analytical predictions match the numerical simulation to <2% across all three scenarios. This is the **"verify, verify, verify, then trust"** pattern the book uses throughout: every numerical computation paired with an independent analytical comparison.

This concludes Part II of the book. Part III (Chapters 6–7 in the print book, or 6, 6B, 7 in our chapter naming) develops parameter estimation and sensitivity analysis on top of this verified SIR_I infrastructure.